# Агент «Куда сходить в Алматы»

**Пайплайн:** sxodim.com → Jina Reader → GPT-5-mini (структурирование) → LlamaIndex RAG → агент

Логика живёт в `src/`, ноутбук её импортирует — так код не дублируется между `.py` и `.ipynb`.


## 0. Настройка

**В Colab** достаточно запустить ячейку ниже — она клонирует репозиторий (ноутбук импортирует из `src/`, поэтому одного `.ipynb` недостаточно), поставит зависимости и возьмёт ключ из панели Secrets (🔑 слева). Добавь туда `OPENAI_API_KEY` до запуска. GPU не нужен — хватит CPU runtime.

**Локально** ячейка ничего не делает: нужен `pip install -r requirements.txt` и `OPENAI_API_KEY` в `.env`.


In [1]:
REPO_URL = "https://github.com/zhadyrazhan/shodim-almaty-agent.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    import subprocess
    from pathlib import Path

    if not Path("shodim-almaty-agent").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    if Path("shodim-almaty-agent").exists():
        os.chdir("shodim-almaty-agent")

    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)

    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Colab: репозиторий и зависимости готовы, ключ загружен")
else:
    print("Локальный запуск — ключ берётся из .env")

Colab: репозиторий и зависимости готовы, ключ загружен


In [2]:
import json

from src import scraper, extract, agent
from src.config import SXODIM_DATA, RAW_DIR, LLM_BACKEND

print("LLM backend:", LLM_BACKEND)

LLM backend: openai


## 1. Парсинг сайта (Jina Reader)

Jina Reader отдаёт готовый markdown по любому URL, поэтому не нужно писать CSS-селекторы под вёрстку, которая может измениться.

In [3]:
written = scraper.scrape_all()
for name, path in written.items():
    print(f"{name}: {path.stat().st_size:,} байт")

fetching afisha: https://sxodim.com/almaty/afisha
  -> afisha.md (66,324 chars)
fetching weekend: https://sxodim.com/almaty/events/weekend
  -> weekend.md (67,553 chars)
fetching places: https://sxodim.com/almaty/places
  -> places.md (60,894 chars)
fetching main: https://sxodim.com/almaty
  -> main.md (87,721 chars)
afisha: 92,695 байт
weekend: 94,393 байт
places: 85,947 байт
main: 117,992 байт


### Фрагмент спарсенных данных

In [4]:
raw = (RAW_DIR / "afisha.md").read_text(encoding="utf-8")
print(raw[:800])

Title: Мероприятия Алматы

URL Source: https://sxodim.com/almaty/afisha

Markdown Content:
[](https://sxodim.com/almaty)

Алматы

Русский Рус

[Қазақша](https://sxodim.com/locale/kk)[Русский](https://sxodim.com/locale/ru)[English](https://sxodim.com/locale/en)

[Билеты](https://sxodim.com/almaty/tickets)[Афиша](https://sxodim.com/almaty/afisha)[Журнал](https://sxodim.com/almaty/journal)[Места](https://sxodim.com/almaty/places)

[![Image 1](https://avatars.mds.yandex.net/get-adfox-content/2462621/260624_adfox_1070705_16070102.6e82ab4e25f7e64070a01903f30fecb3.png/optimize.webp)](https://yandex.ru/adfox/312864/clickURL?ad-session-id=8833751790186769404&adfox-version=1&duid=1790186768275689415&efc=1&esc=0&hash=237533fbd35cf0f6&initial-engine-id=gjj&laas_region_id=ouq&p1=cfgjm&p2=glgo&p5=bjeiiw


## 2. Структурирование в JSON

Спарсенный markdown шумный (меню, реклама, дубли ссылок), поэтому он режется на чанки и передаётся модели со схемой Pydantic — structured output сам приводит всё к типам.

In [5]:
records = extract.extract_all()
print(f"\nвсего записей: {len(records)}")

4 pages -> 58 chunks
  [1/58] afisha.md: +9 (kept 9 unique)
  [2/58] afisha.md: +9 (kept 18 unique)
  [3/58] afisha.md: +2 (kept 20 unique)
  [4/58] afisha.md: +0 (kept 20 unique)
  [5/58] afisha.md: +0 (kept 20 unique)
  [6/58] afisha.md: +0 (kept 20 unique)
  [7/58] afisha.md: +0 (kept 20 unique)
  [8/58] afisha.md: +0 (kept 20 unique)
  [9/58] afisha.md: +0 (kept 20 unique)
  [10/58] afisha.md: +0 (kept 20 unique)
  [11/58] afisha.md: +0 (kept 20 unique)
  [12/58] afisha.md: +0 (kept 20 unique)
  [13/58] afisha.md: +0 (kept 20 unique)
  [14/58] afisha.md: +21 (kept 41 unique)
  [15/58] main.md: +7 (kept 48 unique)
  [16/58] main.md: +9 (kept 51 unique)
  [17/58] main.md: +10 (kept 57 unique)
  [18/58] main.md: +2 (kept 58 unique)
  [19/58] main.md: +0 (kept 58 unique)
  [20/58] main.md: +0 (kept 58 unique)
  [21/58] main.md: +0 (kept 58 unique)
  [22/58] main.md: +1 (kept 59 unique)
  [23/58] main.md: +0 (kept 59 unique)
  [24/58] main.md: +0 (kept 59 unique)
  [25/58] main.md: +0 (

### Структурированный JSON (пример)

In [6]:
data = json.loads(SXODIM_DATA.read_text(encoding="utf-8"))
print(f"записей: {len(data)}\n")
print(json.dumps(data[:3], ensure_ascii=False, indent=2))

записей: 136

[
  {
    "title": "Большой летний фестиваль 2026",
    "kind": "event",
    "description": "4 июля в 17:00, Место проведения будет объявлено позже",
    "category": "фестиваль",
    "url": "https://sxodim.com/almaty/event/bolshoy-letniy-festival-2026",
    "date": "4 июля в 17:00",
    "price": "",
    "address": "",
    "good_for": [
      "компания друзей",
      "семья"
    ]
  },
  {
    "title": "Batyr Amanaty",
    "kind": "event",
    "description": "30 апреля в 19:00, Алматы Арена",
    "category": "концерт",
    "url": "https://sxodim.com/almaty/event/batyr-amanaty-zh-ne-kameraly-orkestr-lken-tribyut-koncert",
    "date": "30 апреля в 19:00",
    "price": "",
    "address": "Almaty Arena, мкр. Нуркент, 7",
    "good_for": [
      "компания друзей",
      "свидание"
    ]
  },
  {
    "title": "Музыкальный фестиваль NonStop Music Fest",
    "kind": "event",
    "description": "3 мая в 19:00, Almaty Arena",
    "category": "фестиваль",
    "url": "https://sxodim.c

In [7]:
from collections import Counter
print("по типам:", dict(Counter(d["kind"] for d in data)))
print("по категориям:", dict(Counter(d["category"] for d in data).most_common(10)))

по типам: {'event': 46, 'place': 84, 'entertainment': 5, 'restaurant': 1}
по категориям: {'театр': 25, 'туры': 11, 'концерт': 7, '': 6, 'Туры от Melon Travel': 5, 'развлечения': 5, 'стендап': 4, 'Туризм': 3, 'Концерты': 3, 'развлечение': 3}


## 3. Агент (LlamaIndex + OpenAI)

Записи индексируются в `VectorStoreIndex`, поверх — query engine с системным промптом гида.

In [8]:
TEST_QUESTIONS = [
    "Куда сходить на выходных?",
    "Посоветуй место для свидания",
    "Куда сводить ребенка?",
    "Какие концерты будут?",
    "Где вкусно поесть?",
]

answers = {}
for q in TEST_QUESTIONS:
    a = agent.ask(q)
    answers[q] = a
    print(f"Q: {q}\nA: {a}\n{'-' * 70}")

Q: Куда сходить на выходных?
A: Отлично — на выходных можно выбрать из нескольких вариантов в афише, в зависимости от настроения и компании. Ниже — пару идей; если скажешь, с кем собираешься идти (семья, друзья, свидание или один), подскажу конкретнее.

- Клуб at-travel — Конные туры и выездки — отличная идея для любителей природы и активного отдыха: подходяще для компании друзей, семьи или в одиночку; учти ограничения по здоровью, весу и возрасту.  
- Активный отдых — подборка мест для спорта и приключений в Алматы, подходит и для компании друзей, и для семьи, и для индивидуальных вылазок.  
- Загородный отдых — если хочешь уехать из города и расслабиться на природе, это хороший вариант для семьи или компании друзей.  
- Ночная жизнь — если планируешь активный вечер с друзьями, посмотри раздел с ночными развлечениями.
----------------------------------------------------------------------
Q: Посоветуй место для свидания
A: Отлично — для свидания в Алматы в афише есть несколько подходящ

## 4. Сохранение примеров диалогов

Обязательный дилеверабл `agent_examples.md` — 5+ примеров.

In [9]:
lines = ["# Примеры диалогов с агентом\n"]
for q, a in answers.items():
    lines.append(f"\n## {q}\n\n{a}\n")

(agent.SXODIM_DATA.parent.parent / "agent_examples.md").write_text(
    "".join(lines), encoding="utf-8"
)
print(f"сохранено {len(answers)} диалогов в agent_examples.md")

сохранено 5 диалогов в agent_examples.md


## 5. Бонус: ORPO — делаем ответы дружелюбнее

Базовый агент отвечает корректно, но суховато. ORPO (Odds Ratio Preference Optimization) объединяет SFT и выравнивание по предпочтениям в один проход: не нужны ни отдельная reward-модель, ни reference-модель в памяти, поэтому всё помещается на бесплатный T4.

**Нужен GPU.** Runtime → Change runtime type → **T4 GPU**, затем Runtime → **Restart session** (смена типа без перезапуска не переносит сессию на GPU). Секции 1-4 выше работают и на CPU: если GPU нет, ячейки ниже сами себя пропустят.

In [ ]:
import subprocess
import torch

HAS_GPU = torch.cuda.is_available()
print("CUDA available:", HAS_GPU)

if HAS_GPU:
    print("GPU:", torch.cuda.get_device_name(0))
    # Ставим только здесь, а не в секции 0: это ~2 ГБ пакетов, которые нужны
    # исключительно для ORPO. На CPU-прогоне секций 1-4 они только мешают.
    print("ставим зависимости для обучения...")
    subprocess.run(
        "pip install -q unsloth unsloth_zoo trl peft accelerate bitsandbytes datasets".split(),
        check=True,
    )
    print("готово")
else:
    print("GPU нет — секция 5 будет пропущена")

### 5.1 Датасет предпочтений

ORPO нужны тройки (prompt, chosen, rejected). Обе стороны генерируются на **одних и тех же** записях афиши: `chosen` — тёплый ответ живым языком, `rejected` — сухая справка списком. Факты одинаковые, отличается только тон, значит модель учится именно стилю, а не содержанию.

Шаг идёт через OpenAI API и GPU не требует.

In [ ]:
if HAS_GPU:
    !python training/build_preference_data.py --n 120

In [ ]:
from pathlib import Path

if HAS_GPU:
    pairs = json.loads(Path("data/preference_data.json").read_text(encoding="utf-8"))
    print(f"пар: {len(pairs)}")
    p = pairs[0]
    print("\nВОПРОС:", p["prompt"])
    print("\n--- CHOSEN (тёплый) ---")
    print(p["chosen"][:500])
    print("\n--- REJECTED (сухой) ---")
    print(p["rejected"][:500])

### 5.2 Ответы ДО обучения

In [ ]:
if HAS_GPU:
    from unsloth import FastLanguageModel

    BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
    model, tok = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=2048, load_in_4bit=True
    )

    def gen(m, t, question, max_new_tokens=220):
        prompt = t.apply_chat_template(
            [{"role": "user", "content": question}],
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = t([prompt], return_tensors="pt").to("cuda")
        out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        return t.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    FastLanguageModel.for_inference(model)
    before = {q: gen(model, tok, q) for q in TEST_QUESTIONS[:3]}
    for q, a in before.items():
        print(f"Q: {q}\nA: {a}\n{'-' * 70}")

### 5.3 Обучение

Следим не только за падением loss, но и за **`rewards/margins`**: именно рост маржи показывает, что модель разводит тёплый и сухой ответы, а не просто подгоняется под оба.

In [ ]:
if HAS_GPU:
    !python training/train_orpo.py --pairs data/preference_data.json --epochs 3

### 5.4 Ответы ПОСЛЕ обучения

In [ ]:
if HAS_GPU:
    from peft import PeftModel

    tuned, tuned_tok = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=2048, load_in_4bit=True
    )
    tuned = PeftModel.from_pretrained(tuned, "outputs/orpo-almaty")
    FastLanguageModel.for_inference(tuned)

    after = {q: gen(tuned, tuned_tok, q) for q in TEST_QUESTIONS[:3]}
    for q, a in after.items():
        print(f"Q: {q}\nA: {a}\n{'-' * 70}")

### 5.5 Сравнение ДО / ПОСЛЕ

Главный артефакт бонусной части: видно ли, что тон стал теплее.

In [ ]:
if HAS_GPU:
    import pandas as pd

    df = pd.DataFrame(
        [{"вопрос": q, "ДО": before[q][:180], "ПОСЛЕ": after[q][:180]} for q in before]
    )
    pd.set_option("display.max_colwidth", 180)
    display(df)

    lines = ["# ORPO: ответы до и после\n"]
    for q in before:
        lines.append(f"\n## {q}\n\n**До:**\n\n{before[q]}\n\n**После:**\n\n{after[q]}\n")
    Path("orpo_examples.md").write_text("".join(lines), encoding="utf-8")
    print("\nсохранено в orpo_examples.md")

## 6. Скачать результаты

Файлы лежат внутри runtime и исчезнут вместе с сессией, поэтому забери их сразу. Сам ноутбук скачивается отдельно: File → Download → Download .ipynb — **после** того, как все ячейки отработали, чтобы выводы сохранились.

In [ ]:
from pathlib import Path

ARTIFACTS = ["agent_examples.md", "data/sxodim_data.json", "orpo_examples.md"]

if IN_COLAB:
    from google.colab import files

    for path in ARTIFACTS:
        # orpo_examples.md only exists if section 5 ran (needs a GPU).
        if Path(path).exists():
            files.download(path)
        else:
            print(f"{path}: пропущен (не создан)")
else:
    for path in ARTIFACTS:
        p = Path(path)
        print(f"{path}: {'есть' if p.exists() else 'НЕТ'}"
              f"{f' ({p.stat().st_size:,} байт)' if p.exists() else ''}")